# 4.04 Vision Transformer：让 Transformer 看图

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/04_transformer/4.04_vit.ipynb)


## 为什么要学

π0 的眼睛是 **SigLIP**（So400m/14），一个 ViT：把 224×224 的图像切成 16×16=256 个 patch，每个 patch 变成一个 token。这些图像 token 和文字 token 拼在一起，送进语言模型。

本课从零实现 ViT，并完成一个"迷你 VLM"任务：**给一张桌面图片和一个颜色，找出该颜色方块的位置**。这就是 VLA 中"视觉-语言对齐（grounding）"能力的最小版本。下一章起，"找到目标"之后就是"输出动作"。参考：[d2l 11.8 ViT](https://zh.d2l.ai/chapter_attention-mechanisms-and-transformers/vision-transformer.html)。

---

## 1. ViT 的结构

```text
图像 (B,3,H,W)
  └─ patch embedding：切成 P×P 小块，每块线性投影到 d 维 ──> (B, N, d)，N = (H/P)(W/P)
  └─ + 位置编码（可学习，(1, N, d)）
  └─ L 个 Transformer Block（双向注意力，没有因果掩码）
  └─ 输出：N 个 patch token（SigLIP 用于 VLM 时直接输出全部 token，不用 CLS）
```

**patch embedding = 线性层作用在展平的 patch 上 = 步长等于核大小的卷积**。1.02 课写过 `patchify`，1.09 课提过卷积写法，这里证明二者等价。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.envs import ToyTableEnv
from vla_ladder.envs.toy_table import COLOR_EN
from vla_ladder.models.transformer import Block, RMSNorm
from vla_ladder.utils import set_seed, get_device, count_params

device = get_device()

### 练习 1：卷积 patch embedding 与"切块 + 线性层"等价

In [ ]:
def patchify(images, P):
    B, C, H, W = images.shape
    x = images.reshape(B, C, H // P, P, W // P, P).permute(0, 2, 4, 1, 3, 5)   # (B, H/P, W/P, C, P, P)
    return x.reshape(B, (H // P) * (W // P), C * P * P)                         # 每个 patch 按 (C, P, P) 展平

def conv_to_linear(conv):
    """把 nn.Conv2d(C, d, P, stride=P) 的权重转换成等价的 nn.Linear(C*P*P, d)"""
    # TODO: conv.weight 形状 (d, C, P, P) -> reshape 成 (d, C*P*P) 作为 linear.weight；bias 直接拷贝
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def patchify(images, P):
    B, C, H, W = images.shape
    x = images.reshape(B, C, H // P, P, W // P, P).permute(0, 2, 4, 1, 3, 5)
    return x.reshape(B, (H // P) * (W // P), C * P * P)

def conv_to_linear(conv):
    """把 nn.Conv2d(C, d, P, stride=P) 的权重转换成等价的 nn.Linear(C*P*P, d)"""
    d = conv.out_channels
    lin = nn.Linear(conv.in_channels * conv.kernel_size[0] * conv.kernel_size[1], d)
    with torch.no_grad():
        lin.weight.copy_(conv.weight.reshape(d, -1))
        lin.bias.copy_(conv.bias)
    return lin

In [ ]:
# ✅ 自测
conv = nn.Conv2d(3, 64, kernel_size=8, stride=8)
img = torch.randn(2, 3, 48, 48)
via_conv = conv(img).flatten(2).transpose(1, 2)               # (B, 36, 64)
via_linear = conv_to_linear(conv)(patchify(img, 8))
assert torch.allclose(via_conv, via_linear, atol=1e-5)
print("通过 ✅ 卷积 patch embedding ≡ 切块 + 线性层")

> 注意：这里 patch 内部的展平顺序是 `(C, P, P)`，与 1.02 课的 `(P, P, C)` 不同。**这正是 1.02 思考题说的情况**：只要和预训练权重的约定一致就行，不一致时不会报错，模型却会变差。

---

## 2. 任务：看图找方块

桌面上有 3 个不同颜色的方块。给定一个颜色（红/绿/蓝/黄之一，且保证在图中），输出该颜色方块中心的坐标 $(x,y)$。

In [ ]:
def make_grounding_data(n, seed):
    env = ToyTableEnv(num_envs=n, seed=seed)
    obs = env.reset()
    images = torch.tensor(obs["image"])                                    # (n, 3, 48, 48)
    color_id = torch.tensor(env.colors[np.arange(n), env.target_idx])      # 目标颜色编号 0..3
    target_xy = torch.tensor(env.target, dtype=torch.float32)              # (n, 2)
    return images, color_id, target_xy

train_imgs, train_col, train_xy = make_grounding_data(8000, seed=0)
test_imgs, test_col, test_xy = make_grounding_data(1000, seed=1)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for i, ax in enumerate(axes):
    ax.imshow(train_imgs[i].permute(1, 2, 0), extent=(-1, 1, -1, 1))
    ax.plot(*train_xy[i], "kx", ms=10, mew=2); ax.set_title(f"find: {COLOR_EN[train_col[i]]}"); ax.axis("off")
plt.show()

### 模型：ViT + 一个"查询 token"

把颜色编号变成一个可学习的 embedding（相当于只有 4 个词的"语言"），作为**额外的 token 拼在图像 token 后面**，一起过 Transformer（双向注意力）。最后读出这个查询 token 的输出，用线性层预测坐标。

这正是 VLM 的工作方式：**文字 token 和图像 token 在同一个 Transformer 中交互**。查询 token 会学会"去注意那个颜色的方块所在的 patch"。

### 练习 2：实现 GroundingViT

In [ ]:
class GroundingViT(nn.Module):
    def __init__(self, image_size=48, patch=8, dim=128, depth=4, n_heads=4, n_colors=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        n_patches = (image_size // patch) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, dim) * 0.02)
        self.color_embed = nn.Embedding(n_colors, dim)           # "语言"：颜色 -> 向量
        self.blocks = nn.ModuleList([Block(dim, n_heads, dim // n_heads, 4 * dim) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, 2)

    def forward(self, images, color_id):
        # TODO:
        # 1) img_tok = patch_embed(images).flatten(2).transpose(1, 2) + pos_embed      (B, N, dim)
        # 2) q_tok = color_embed(color_id)[:, None, :]                                 (B, 1, dim)
        # 3) x = cat([img_tok, q_tok], dim=1)，依次过 blocks（不需要掩码：全双向）
        # 4) 取最后一个 token：head(norm(x)[:, -1])                                    (B, 2)
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class GroundingViT(nn.Module):
    def __init__(self, image_size=48, patch=8, dim=128, depth=4, n_heads=4, n_colors=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        n_patches = (image_size // patch) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, dim) * 0.02)
        self.color_embed = nn.Embedding(n_colors, dim)
        self.blocks = nn.ModuleList([Block(dim, n_heads, dim // n_heads, 4 * dim) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, 2)

    def forward(self, images, color_id):
        img_tok = self.patch_embed(images).flatten(2).transpose(1, 2) + self.pos_embed
        q_tok = self.color_embed(color_id)[:, None, :]
        x = torch.cat([img_tok, q_tok], dim=1)
        for blk in self.blocks:
            x, _ = blk(x)
        return self.head(self.norm(x)[:, -1])

In [ ]:
# ✅ 自测：形状
set_seed(0)
model = GroundingViT().to(device)
out = model(train_imgs[:5].to(device), train_col[:5].to(device))
assert out.shape == (5, 2)
print(f"通过 ✅ 参数量 {count_params(model) / 1e6:.2f} M")

### 训练（GPU 约 1 分钟，CPU 约 4~6 分钟）

In [ ]:
def train_grounding(model, steps=4000, B=128, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    imgs, cols, xys = train_imgs.to(device), train_col.to(device), train_xy.to(device)
    t0 = time.time()
    for step in range(steps):
        i = torch.randint(0, len(imgs), (B,), device=device)
        loss = F.mse_loss(model(imgs[i], cols[i]), xys[i])
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % 500 == 0 or step == steps - 1:
            print(f"step {step:5d}  loss {loss.item():.4f}  ({time.time() - t0:.0f}s)")

@torch.no_grad()
def test_error(model):
    model.eval()
    pred = model(test_imgs.to(device), test_col.to(device)).cpu()
    model.train()
    return (pred - test_xy).norm(dim=-1)

train_grounding(model)
err = test_error(model)
print(f"测试集平均定位误差 {err.mean():.3f}（方块边长 0.24，一个像素约 0.042）；误差 < 0.1 的比例 {(err < 0.1).float().mean():.2f}")
assert (err < 0.1).float().mean() > 0.7

### 看看查询 token 在"看"哪里

手动计算最后一层里查询 token 对各个图像 patch 的注意力权重（各头平均），画成 6×6 的热力图：

In [ ]:
@torch.no_grad()
def query_attention_map(model, image, color_id):
    img_tok = model.patch_embed(image[None]).flatten(2).transpose(1, 2) + model.pos_embed
    x = torch.cat([img_tok, model.color_embed(color_id[None])[:, None]], 1)
    for blk in model.blocks[:-1]:
        x, _ = blk(x)
    attn = model.blocks[-1].self_attn
    q, k, _ = attn.qkv(model.blocks[-1].input_layernorm(x))           # (1, h, N+1, hd)
    w = torch.softmax(q[:, :, -1:] @ k.transpose(-2, -1) / math.sqrt(attn.head_dim), -1)   # 查询 token 那一行
    return w[0, :, 0, :-1].mean(0).reshape(6, 6)

model.eval()
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for j in range(4):
    img, col = test_imgs[j].to(device), test_col[j].to(device)
    axes[0, j].imshow(test_imgs[j].permute(1, 2, 0), extent=(-1, 1, -1, 1)); axes[0, j].set_title(f"find: {COLOR_EN[col]}"); axes[0, j].axis("off")
    with torch.no_grad():
        pred = model(img[None], col[None])[0].cpu()
    axes[0, j].plot(*pred, "k+", ms=12, mew=2)
    axes[1, j].imshow(query_attention_map(model, img, col).cpu(), cmap="hot", extent=(-1, 1, -1, 1)); axes[1, j].set_title("query attention"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

查询 token 的注意力集中在目标方块附近的 patch 上：**它学会了"按语言去看图"**。

---

## 3. 从 GroundingViT 到 VLA

| 本课 | 真实 VLA（π0） |
|---|---|
| 4 种颜色的 embedding | 25.7 万词表的 Gemma embedding + 一整句指令 |
| 自己从零训练的 4 层 ViT | 预训练的 SigLIP（4 亿参数），在海量图文对上训练过 |
| 图像 token + 1 个查询 token，双向注意力 | 图像 token + 文字 token（前缀，双向） |
| 输出：方块坐标 | 输出：50 步动作块（由动作专家通过流匹配生成） |

**预训练的价值**：本课的模型只认识 4 种颜色、一种场景。SigLIP + Gemma 在互联网数据上预训练过，"认识"成千上万的物体和概念。微调时，VLA 把这些知识迁移到机器人控制上，这是 VLA 相比 ACT/Diffusion Policy 泛化能力强的根本原因。

## 小结

- ViT：patch embedding（≡ 步长为 P 的卷积）+ 位置编码 + 双向 Transformer。
- 把"语言 token"和图像 token 拼在一起做注意力，就是 VLM 的核心机制；查询 token 学会了按语言注意图像的特定区域。
- 🎉 **第 4 章完成**：你已经能从零写出 VLA 的"V"和"L"。接下来学习"A"：怎么生成动作。